# Class Test Data

## Objective

Create a one-row-per-match dataset designed for the `PredictGame` Poisson simulation class in `3.SimulationStudy/PoissonClass.ipynb`.

The class needs the model inputs from each team's perspective. Since each row includes both teams, this notebook stores home-side and away-side inputs separately, even though World Cup matches are neutral-site games.

The Brazil-Germany 2014 semi-final (`M-2014-61`) is dropped to match the other model tests.


## Libraries

In [ ]:
library(tidyverse)
library(here)

## Source data

The data sources mirror the model-selection notebooks:

- `FullTeamGames.rds` from `WorldCups.ipynb`
- `ELOSScores.rds` from `ELOScores.ipynb`
- `ManagerWC_history.rds` from `ManagerExperience.ipynb`
- `PlayerExperience.rds` from `PlayerExperience.ipynb`
- `DistanceFromHome.rds` from `DistanceFromHome.ipynb`

In [ ]:
Games <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "FullTeamGames.rds"))
ELO <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "ELOSScores.rds"))
ManagerHistory <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "ManagerWC_history.rds"))
PlayerExperience <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "PlayerExperience.rds"))
DistanceFromHome <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "DistanceFromHome.rds"))

## Team-match features

Build the same team-perspective inputs used by the Poisson model, including opponent manager and player-experience variables. This intermediate table still has two rows per game.

In [ ]:
completed_tournaments <- c("WC-2010", "WC-2014", "WC-2018", "WC-2022")
brazil_germany_match_id <- "M-2014-61"

ELOByTournament <- ELO %>%
  pivot_longer(
    cols = starts_with("WC_"),
    names_to = "tournament_year",
    values_to = "elo"
  ) %>%
  mutate(tournament_id = str_replace(tournament_year, "_", "-")) %>%
  select(team, tournament_id, elo)

TeamManagerFeatures <- ManagerHistory %>%
  select(
    tournament_id,
    team_id,
    team_prior_world_cups_coached = prior_world_cups_coached
  )

OpponentManagerFeatures <- ManagerHistory %>%
  select(
    tournament_id,
    opponent_id = team_id,
    opp_prior_world_cups_coached = prior_world_cups_coached
  )

TeamPlayerFeatures <- PlayerExperience %>%
  select(
    tournament_id,
    team_id,
    avg_age,
    team_players_with_multiple_prior_wcs = players_with_multiple_prior_wcs
  )

OpponentPlayerFeatures <- PlayerExperience %>%
  select(
    tournament_id,
    opponent_id = team_id,
    opp_avg_age = avg_age,
    opp_players_with_multiple_prior_wcs = players_with_multiple_prior_wcs
  )

TeamDistance <- DistanceFromHome %>%
  select(
    tournament_id,
    team_id,
    distance_from_host_km
  )

OpponentDistance <- DistanceFromHome %>%
  select(
    tournament_id,
    opponent_id = team_id,
    opp_distance_from_host_km = distance_from_host_km
  )

TeamMatchRows <- Games %>%
  filter(
    tournament_id %in% completed_tournaments,
    match_id != brazil_germany_match_id
  ) %>%
  left_join(
    ELOByTournament %>% rename(team_name = team, team_ELO = elo),
    by = c("team_name", "tournament_id")
  ) %>%
  left_join(
    ELOByTournament %>% rename(opponent_name = team, opponent_ELO = elo),
    by = c("opponent_name", "tournament_id")
  ) %>%
  left_join(TeamManagerFeatures, by = c("tournament_id", "team_id")) %>%
  left_join(OpponentManagerFeatures, by = c("tournament_id", "opponent_id")) %>%
  left_join(TeamPlayerFeatures, by = c("tournament_id", "team_id")) %>%
  left_join(OpponentPlayerFeatures, by = c("tournament_id", "opponent_id")) %>%
  left_join(TeamDistance, by = c("tournament_id", "team_id")) %>%
  left_join(OpponentDistance, by = c("tournament_id", "opponent_id")) %>%
  mutate(
    ELO_diff = team_ELO - opponent_ELO,
    SouthAfrica = if_else(tournament_id == "WC-2010", 1, 0)
  )

TeamMatchRows %>%
  summarise(
    rows = n(),
    matches = n_distinct(match_id),
    missing_values = sum(is.na(across(everything())))
  )

## One row per game

The first team row after sorting by `team_id` becomes the `home` side. This is only a stable label for the class inputs; it does not mean the team was a true home team.

In [ ]:
ClassTestData <- TeamMatchRows %>%
  arrange(tournament_id, match_id, team_id) %>%
  group_by(match_id) %>%
  mutate(match_side = row_number()) %>%
  ungroup() %>%
  filter(match_side == 1) %>%
  transmute(
    tournament_id,
    match_id,
    stage_name,
    group_stage,
    city_name,
    match_date,
    match_time,
    SouthAfrica,

    home_team_id = team_id,
    home_team = team_name,
    home_goals = goals_for,
    home_ELO = team_ELO,
    home_prior_world_cups_coached = team_prior_world_cups_coached,
    home_players_with_multiple_prior_wcs = team_players_with_multiple_prior_wcs,
    home_avg_age = avg_age,
    home_avg_age_squared = avg_age^2,
    home_distance_from_host_km = distance_from_host_km,

    away_team_id = opponent_id,
    away_team = opponent_name,
    away_goals = goals_against,
    away_ELO = opponent_ELO,
    away_prior_world_cups_coached = opp_prior_world_cups_coached,
    away_players_with_multiple_prior_wcs = opp_players_with_multiple_prior_wcs,
    away_avg_age = opp_avg_age,
    away_avg_age_squared = opp_avg_age^2,
    away_distance_from_host_km = opp_distance_from_host_km,

    home_ELO_diff = home_ELO - away_ELO,
    away_ELO_diff = away_ELO - home_ELO,

    home_class_EloHome = home_ELO,
    home_class_EloAway = away_ELO,
    home_class_opp_wc_coach = away_prior_world_cups_coached,
    home_class_opp_players_multiple_WCs = away_players_with_multiple_prior_wcs,
    home_class_AVGage = home_avg_age,
    home_class_opp_AVGage = away_avg_age,
    home_class_Distance_from_host_km = home_distance_from_host_km,

    away_class_EloHome = away_ELO,
    away_class_EloAway = home_ELO,
    away_class_opp_wc_coach = home_prior_world_cups_coached,
    away_class_opp_players_multiple_WCs = home_players_with_multiple_prior_wcs,
    away_class_AVGage = away_avg_age,
    away_class_opp_AVGage = home_avg_age,
    away_class_Distance_from_host_km = away_distance_from_host_km
  )

head(ClassTestData)

## Validation

In [ ]:
ClassTestData %>%
  summarise(
    rows = n(),
    unique_matches = n_distinct(match_id),
    has_brazil_germany_2014 = any(match_id == brazil_germany_match_id),
    missing_values = sum(is.na(across(everything()))),
    duplicated_matches = sum(duplicated(match_id))
  )

ClassTestData %>%
  count(tournament_id)

ClassTestData %>%
  filter(
    tournament_id == "WC-2014",
    (home_team == "Brazil" & away_team == "Germany") |
      (home_team == "Germany" & away_team == "Brazil")
  )

## Save

In [ ]:
saveRDS(ClassTestData, here("1.DataCleaning-R", "Data", "RDS", "ClassTestData.rds"))
write_csv(ClassTestData, here("1.DataCleaning-R", "Data", "CSV", "ClassTestData.csv"))